# 设备可能出问题时，什么时候值得停机检查？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'probability_model': 'calibrated', 'validation_brier': {'raw': 0.0803071585982848, 'calibrated': 0.04474469729236851}, 'test_brier': 0.04872725557281508, 'action_threshold': 0.0784313725490196, 'capacity': 40, 'costs_under_assumed_effectiveness': {'none': {'total_cost': 29400.0, 'inspections': 0, 'uninspected_failures': 49}, 'threshold_0_5': {'total_cost': 17030.0, 'inspections': 35, 'uninspected_failures': 22}, 'net_benefit': {'total_cost': 15700.0, 'inspections': 40, 'uninspected_failures': 19}}, 'positive_benefit_without_capacity': 56}


## 1. 补全：盈亏平衡
在理论阈值处验证预期避免损失恰等于检查成本。

In [2]:
assert np.isclose(threshold*failure_loss*effectiveness,inspection_cost)
assert (queue.loc[queue['inspect'],'expected_net_benefit']>=-1e-10).all()
print(threshold)

0.0784313725490196


## 2. 修改：容量减半
把容量改为 20，核对检查不超过 20 台，并输出情景成本。

In [3]:
small=choose_inspections(p,threshold,20)
assert small.sum()<=20
assert np.all(~small|selected)
print(policy_cost(small,y_test.astype(bool)))

{'total_cost': 20510.0, 'inspections': 20, 'uninspected_failures': 30}


## 3. 迁移：检查效果只剩 50%
仅重新计算理论阈值与检查名单，说明为什么阈值升高；不要继续使用旧效果结算函数。

In [4]:
weaker=0.5
new_threshold=inspection_cost/(failure_loss*weaker)
new_mask=choose_inspections(p,new_threshold,capacity)
new_cost=float(np.sum(inspection_cost*new_mask+failure_loss*y_test*(1-weaker*new_mask)))
assert new_threshold>threshold
print({'新阈值':new_threshold,'检查数':int(new_mask.sum()),'新效果假设成本':new_cost})

{'新阈值': 0.13333333333333333, '检查数': 40, '新效果假设成本': 22000.0}
